In [ ]:
import pandas as pd
from tqdm import tqdm

k = 7
data_folder = '2WikiMultihopQA'
data_extenstion = '-full-list'
llm_provider = 'ollama'  # 'openai' or 'ollama' or 'huggingface'
mode = 'rag'  # 'related_data' or 'rag+related_data' or 'rag'

## RAG


In [ ]:
from RAG import RAG

rag = RAG(data_folder=data_folder,
          embedding_model='all-mpnet-base-v2',
          chromadb_extension_name=data_extenstion,
          mode=mode,
          llm_provider=llm_provider,
          llm_model="default")
rag.create_chromadb()
# rag.start(k=k)

In [ ]:
results = rag.load_results()
questions = rag.load_questions(limit=True, count=100)[:3]
total_question = len(questions)
display(questions)

In [ ]:
for index, row in tqdm(questions.iterrows(),
                       desc='Getting results',
                       dynamic_ncols=True,
                       total=total_question):

    res = rag.run(row['question'])['result']

    if relationships:
        retrieval_context = [rag.related_data] + rag.get_context(
            row['question'])
    else:
        retrieval_context = rag.get_context(row['question'])

    if res == "Error: The model name or API request is invalid.":
        print(f"Error: {res}")
        continue

    new_row = pd.DataFrame({
        'id': [index],
        'type': [row['type']],
        'question': [row['question']],
        'answer': [row['answer']],
        'result': [res],
        'k': [k],
        'retrieval_context': [retrieval_context],
        'llm_model': [rag.llm_model],
    })
    results = pd.concat([results, new_row], ignore_index=True)

rag.save_results(results=results)
results

## ChatGPT


In [ ]:
from LLM import ChatGPT

chat_gpt = ChatGPT(data_folder=data_folder, model="gpt-4o", temperature=0.7)

In [ ]:
chat_gpt.run()

In [ ]:
chat_gpt.save_results()

## Deep Eval


In [ ]:
from DeepEval import Deep_Eval
# [
#     'Contextual Precision', 'Contextual Relevancy', 'Ragas', 'Faithfulness',
#     'Answer Relevancy'
# ]
deep_eval = Deep_Eval(data_folder=data_folder,
                      data_extension_name=data_extenstion,
                      relationships=relationships,
                      test_mode=False,
                      metrics=[
                          'Contextual Precision', 'Contextual Relevancy',
                          'Ragas', 'Faithfulness', 'Answer Relevancy'
                      ],
                      k=k)

In [ ]:
deep_eval.run()
deep_eval.show_results()

In [ ]:
deep_eval.save_results()